# Qwen 1.5B RAG Notebook

Lucas Harrich

This notebook uses a RIS-based Q&A questions, retrieves the best matching entries, and then answers with a conservative retrieval-first approach.

What it does:
- it loads `rag_data/ris_kstg_qa_corpus.jsonl`
- it builds embeddings for the RIS-based Q&A corpus
- it retrieves the best matching entries for each prompt
- it uses Qwen only as a light rewrite step when needed
- if the model output looks weak, it falls back to the best retrieved RIS answer
- it writes `RAG_qwen_1_5b.csv`


In [7]:
import subprocess
import sys

packageList = ["sentence-transformers", "mlx-lm", "pandas", "numpy", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packageList])


0

In [ ]:
# Imports and settings
import csv
import json
import re
from pathlib import Path as pathClass

import numpy as numpyLib
import pandas as pandasLib
import torch
from IPython.display import display as showTable
from mlx_lm import generate as generateTextFunc
from mlx_lm import load as loadModelFunc
from mlx_lm.sample_utils import make_logits_processors as makeLogitsProcessorsFunc
from mlx_lm.sample_utils import make_sampler as makeSamplerFunc
from sentence_transformers import SentenceTransformer as sentenceTransformerClass

inputPath = pathClass("dataset_clean.csv")
corpusPath = pathClass("rag_data/ris_kstg_qa_corpus.jsonl")
embeddingCachePath = pathClass("rag_data/ris_kstg_qa_corpus_embeddings.npy")
outputPath = pathClass("RAG_qwen_1_5b.csv")
debugOutputPath = pathClass("submission_qwen_1_5b_rag_ris_debug.csv")

useSample = False
sampleSize = 5
topK = 5
writeDebugOutput = False

embeddingModelName = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
generatorModelName = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

maxNewTokens = 80
generationTemperature = 0.1
generationTopP = 0.9
generationTopK = 20
repetitionPenalty = 1.10
repetitionContextSize = 64
directAnswerThreshold = 0.72

systemPrompt = (
    "Beantworte die oesterreichische Steuerfrage auf Deutsch in hoechstens zwei kurzen Saetzen. "
    "Nutze nur den gegebenen RIS-Kontext. "
    "Wenn eine Aussage klar passt, gib sie moeglichst wortnah wieder. "
    "Kein Vorspann, keine Meta-Saetze, keine Wiederholung der Frage, keine erfundenen Fakten."
)

requiredInputColumns = {"id", "prompt"}
requiredCorpusColumns = {"chunk_id", "topic", "source_title", "source_url", "question", "answer", "search_text"}


In [9]:
# Load and validate data
sourceTable = pandasLib.read_csv(inputPath, encoding="utf-8-sig")
missingInputColumns = requiredInputColumns - set(sourceTable.columns)
if missingInputColumns:
    raise ValueError(f"Missing columns: {sorted(missingInputColumns)}")

sourceTable["prompt"] = sourceTable["prompt"].fillna("").astype(str)
workingTable = sourceTable.head(sampleSize).copy() if useSample else sourceTable.copy()

corpusRowList = []
with corpusPath.open("r", encoding="utf-8") as sourceFile:
    for line in sourceFile:
        if not line.strip():
            continue
        row = json.loads(line)
        missingCorpusColumns = requiredCorpusColumns - set(row)
        if missingCorpusColumns:
            raise ValueError(f"Missing corpus columns: {sorted(missingCorpusColumns)}")
        corpusRowList.append(row)

if not corpusRowList:
    raise ValueError("Corpus is empty.")

print(f"prediction rows: {len(workingTable)}")
print(f"corpus rows: {len(corpusRowList)}")
showTable(workingTable.head())


prediction rows: 643
corpus rows: 138


,id,prompt
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ..."
2,CORP-TAX-003,"Welche Körperschaften sind verpflichtet, sämtl..."
3,CORP-TAX-004,Wie wird eine Dividende einer österreichischen...
4,CORP-TAX-005,Was unterscheidet die steuerliche Behandlung e...


In [10]:
# Build or load corpus embeddings
def pickEmbeddingDevice():
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

embeddingDevice = pickEmbeddingDevice()
embedder = sentenceTransformerClass(embeddingModelName, device=embeddingDevice)

corpusSearchTextList = [row["search_text"] for row in corpusRowList]

if embeddingCachePath.exists():
    corpusEmbeddingMatrix = numpyLib.load(embeddingCachePath)
    if len(corpusEmbeddingMatrix) != len(corpusSearchTextList):
        corpusEmbeddingMatrix = embedder.encode(
            corpusSearchTextList,
            normalize_embeddings=True,
            show_progress_bar=True
        )
        numpyLib.save(embeddingCachePath, corpusEmbeddingMatrix)
else:
    corpusEmbeddingMatrix = embedder.encode(
        corpusSearchTextList,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    embeddingCachePath.parent.mkdir(parents=True, exist_ok=True)
    numpyLib.save(embeddingCachePath, corpusEmbeddingMatrix)

print(f"embedding model: {embeddingModelName}")
print(f"embedding device: {embeddingDevice}")
print(f"embedding shape: {corpusEmbeddingMatrix.shape}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6399.79it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
embedding device: mps
embedding shape: (138, 384)


In [11]:
# Load the Qwen generator and helper functions
generatorModel, generatorTokenizer = loadModelFunc(generatorModelName)

sampler = makeSamplerFunc(
    temp=generationTemperature,
    top_p=generationTopP,
    top_k=generationTopK,
)

logitsProcessors = makeLogitsProcessorsFunc(
    repetition_penalty=repetitionPenalty,
    repetition_context_size=repetitionContextSize,
)

stopwordSet = {
    "die", "der", "das", "und", "oder", "ist", "sind", "eine", "einer", "eines", "einem",
    "einen", "ein", "im", "in", "den", "dem", "des", "zu", "auf", "von", "mit", "fuer",
    "für", "nach", "bei", "als", "nur", "auch", "wie", "was", "wann", "welche", "welcher",
    "welches", "wird", "werden", "gilt", "gelten", "kann", "koennen", "können", "nicht"
}

def cleanText(value):
    return " ".join(str(value).strip().split())

def normalizeWordList(text):
    tokenList = re.findall(r"[A-Za-zÄÖÜäöüß0-9]+", str(text).lower())
    return [token for token in tokenList if len(token) >= 4 and token not in stopwordSet]

corpusTokenSetList = [set(normalizeWordList(row["search_text"])) for row in corpusRowList]

def retrievePassages(questionText):
    queryEmbedding = embedder.encode(
        [questionText],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    denseScoreArray = corpusEmbeddingMatrix @ queryEmbedding
    queryTokenSet = set(normalizeWordList(questionText))

    lexicalScoreList = []
    for tokenSet in corpusTokenSetList:
        if not queryTokenSet:
            lexicalScoreList.append(0.0)
            continue
        lexicalScoreList.append(len(queryTokenSet & tokenSet) / len(queryTokenSet))

    lexicalScoreArray = numpyLib.array(lexicalScoreList, dtype=float)
    hybridScoreArray = (0.78 * denseScoreArray) + (0.22 * lexicalScoreArray)
    bestIndexList = numpyLib.argsort(hybridScoreArray)[-topK:][::-1]

    resultList = []
    for idx in bestIndexList:
        row = dict(corpusRowList[int(idx)])
        row["dense_score"] = float(denseScoreArray[int(idx)])
        row["lexical_score"] = float(lexicalScoreArray[int(idx)])
        row["score"] = float(hybridScoreArray[int(idx)])
        resultList.append(row)
    return resultList

def buildUserPrompt(questionText, retrievedRowList):
    contextBlockList = []
    for itemIndex, row in enumerate(retrievedRowList, start=1):
        contextBlockList.append(
            f"[{itemIndex}] Aehnliche Frage: {row['question']}\n"
            f"Passende Antwort: {row['answer']}\n"
            f"Quelle: {row['source_title']}\n"
            f"URL: {row['source_url']}"
        )

    contextText = "\n\n".join(contextBlockList)
    return (
        f"Kontext aus RIS:\n{contextText}\n\n"
        f"Frage:\n{questionText}\n\n"
        "Gib eine kurze Antwort, die eng am Kontext bleibt."
    )

def generateAnswer(questionText, retrievedRowList):
    promptMessages = [
        {"role": "system", "content": systemPrompt},
        {"role": "user", "content": buildUserPrompt(questionText, retrievedRowList)}
    ]

    fullPrompt = generatorTokenizer.apply_chat_template(
        promptMessages,
        tokenize=False,
        add_generation_prompt=True
    )

    answerText = generateTextFunc(
        generatorModel,
        generatorTokenizer,
        prompt=fullPrompt,
        max_tokens=maxNewTokens,
        sampler=sampler,
        logits_processors=logitsProcessors,
        verbose=False
    )
    return cleanText(answerText)

def looksWeak(answerText, retrievedRowList):
    text = cleanText(answerText)
    lowered = text.lower()
    if not text:
        return True
    if len(text) < 25:
        return True
    if "kontext" in lowered or "ris" in lowered or "frage" in lowered:
        return True
    if "?" in text:
        return True
    answerTokenSet = set(normalizeWordList(text))
    retrievedTokenSet = set()
    for row in retrievedRowList:
        retrievedTokenSet |= set(normalizeWordList(row["answer"]))
    if answerTokenSet and len(answerTokenSet & retrievedTokenSet) <= 1:
        return True
    return False

def chooseFinalAnswer(questionText, retrievedRowList):
    bestRetrievedAnswer = cleanText(retrievedRowList[0]["answer"])
    bestRetrievedScore = float(retrievedRowList[0]["score"])

    if bestRetrievedScore >= directAnswerThreshold:
        return bestRetrievedAnswer, "direct_retrieval"

    generatedAnswer = generateAnswer(questionText, retrievedRowList)
    if looksWeak(generatedAnswer, retrievedRowList):
        return bestRetrievedAnswer, "retrieval_fallback"

    return generatedAnswer, "generator"

print(f"generator model: {generatorModelName}")


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 155344.59it/s]


generator model: mlx-community/Qwen2.5-1.5B-Instruct-4bit


In [12]:
# Run RAG and save the output files
answerRowList = []
debugRowList = []

for rowIndex, row in enumerate(workingTable.itertuples(index=False), start=1):
    questionText = cleanText(row.prompt)
    retrievedRowList = retrievePassages(questionText)
    answerText, answerMode = chooseFinalAnswer(questionText, retrievedRowList)

    answerRowList.append({"id": row.id, "answer": answerText})
    debugRowList.append({
        "id": row.id,
        "answer": answerText,
        "mode": answerMode,
        "retrieved_questions": " | ".join(item["question"] for item in retrievedRowList),
        "retrieved_answers": " | ".join(item["answer"] for item in retrievedRowList),
        "retrieved_urls": " | ".join(item["source_url"] for item in retrievedRowList),
        "retrieved_scores": " | ".join(f"{item['score']:.4f}" for item in retrievedRowList),
    })

    print(f"{rowIndex}/{len(workingTable)}: {row.id} [{answerMode}]")

answerTable = pandasLib.DataFrame(answerRowList, columns=["id", "answer"])
answerTable.to_csv(outputPath, index=False, encoding="utf-8")

if writeDebugOutput:
    debugTable = pandasLib.DataFrame(
        debugRowList,
        columns=["id", "answer", "mode", "retrieved_questions", "retrieved_answers", "retrieved_urls", "retrieved_scores"]
    )
    debugTable.to_csv(debugOutputPath, index=False, encoding="utf-8")
    print(f"saved debug csv: {debugOutputPath.resolve()}")
    showTable(debugTable.head())

print(f"saved submission: {outputPath.resolve()}")
showTable(answerTable.head())


1/643: CORP-TAX-001 [direct_retrieval]
2/643: CORP-TAX-002 [generator]
3/643: CORP-TAX-003 [generator]
4/643: CORP-TAX-004 [generator]
5/643: CORP-TAX-005 [generator]
6/643: CORP-TAX-006 [generator]
7/643: CORP-TAX-007 [generator]
8/643: CORP-TAX-008 [direct_retrieval]
9/643: CORP-TAX-009 [generator]
10/643: CORP-TAX-010 [retrieval_fallback]
11/643: CORP-TAX-011 [retrieval_fallback]
12/643: CORP-TAX-012 [generator]
13/643: CORP-TAX-013 [generator]
14/643: CORP-TAX-014 [generator]
15/643: CORP-TAX-015 [generator]
16/643: CORP-TAX-016 [generator]
17/643: CORP-TAX-017 [generator]
18/643: CORP-TAX-018 [generator]
19/643: CORP-TAX-019 [generator]
20/643: CORP-TAX-020 [generator]
21/643: CORP-TAX-021 [generator]
22/643: CORP-TAX-022 [generator]
23/643: CORP-TAX-023 [generator]
24/643: CORP-TAX-024 [generator]
25/643: CORP-TAX-025 [direct_retrieval]
26/643: CORP-TAX-026 [generator]
27/643: CORP-TAX-027 [generator]
28/643: CORP-TAX-028 [retrieval_fallback]
29/643: CORP-TAX-029 [generator]
30/6

,id,answer
0,CORP-TAX-001,Der Körperschaftsteuer ist das Einkommen zugru...
1,CORP-TAX-002,Bei einem zinslosen Darlehen an den Gesellscha...
2,CORP-TAX-003,"Die Körperschaften, die sämtliche Einkünfte de..."
3,CORP-TAX-004,Ein Dividende von einer österreichischen Tocht...
4,CORP-TAX-005,Die steuerliche Behandlung einer offenen von e...
